# UK House Price Prediction - Phase 1: EDA & Preprocessing

**Project Goal:** Build a linear regression model to predict UK house prices using regional and temporal data

**Dataset:** UK House Prices (2015-2025)  
**Observations:** 6,336 records  
**Features:** Time (Year, Month), Region, Property Type, Sales Volume, Median Price, Price per m²

**Phase 1 Objectives:**
1. Load and understand the UK house prices dataset
2. Perform temporal and regional analysis
3. Analyze target variable distribution (AveragePrice_GBP)
4. Explore relationships between features
5. Identify preprocessing needs

---

## Setup & Imports

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import skew, kurtosis

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

: 

## 1. Load Dataset

Load the UK house prices data.

In [ ]:
# Load the dataset
df = pd.read_csv('data/uk_house_prices_raw.csv')

print("=" * 80)
print("DATASET LOADED")
print("=" * 80)
print(f"\nShape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Display first and last few rows
print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

In [ ]:
# Data types and info
print("=" * 80)
print("COLUMN INFORMATION")
print("=" * 80)
df.info()

## 2. Data Quality Check

Check for missing values, duplicates, and data quality issues.

In [ ]:
# Missing values analysis
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})

if missing.sum() > 0:
    print("\nFeatures with missing values:\n")
    print(missing_df[missing_df['Missing Count'] > 0])
else:
    print("\n✓ No missing values detected! Clean dataset.")

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

if duplicates > 0:
    print("⚠ Warning: Duplicates found. Consider removing.")
else:
    print("✓ No duplicates found.")

In [ ]:
# Unique values per column
print("\n" + "=" * 80)
print("UNIQUE VALUES PER COLUMN")
print("=" * 80)

for col in df.columns:
    n_unique = df[col].nunique()
    print(f"{col:20s}: {n_unique:5d} unique values", end="")
    
    # Show categories for low-cardinality features
    if n_unique < 20 and col not in ['RecordID']:
        print(f" → {sorted(df[col].unique().tolist())}")
    else:
        print()

## 3. Target Variable Analysis: AveragePrice_GBP

Deep dive into the distribution and characteristics of our target variable.

In [ ]:
# Target variable statistics
print("=" * 80)
print("TARGET VARIABLE: AveragePrice_GBP")
print("=" * 80)
print("\nDescriptive Statistics:")
print(df['AveragePrice_GBP'].describe())

print(f"\n{'Skewness:':<20} {df['AveragePrice_GBP'].skew():.4f}")
print(f"{'Kurtosis:':<20} {df['AveragePrice_GBP'].kurtosis():.4f}")
print(f"{'Range:':<20} £{df['AveragePrice_GBP'].min():,.0f} - £{df['AveragePrice_GBP'].max():,.0f}")
print(f"{'IQR:':<20} £{df['AveragePrice_GBP'].quantile(0.75) - df['AveragePrice_GBP'].quantile(0.25):,.0f}")

print("\n--- Interpretation ---")
skewness = df['AveragePrice_GBP'].skew()
if skewness > 1:
    print("⚠ HIGHLY right-skewed distribution (skew > 1)")
    print("  → Log transformation STRONGLY recommended for linear regression")
    print("  → High-price properties are pulling the mean upward")
elif skewness > 0.5:
    print("⚠ Moderate right skew detected")
    print("  → Log transformation recommended")
else:
    print("✓ Distribution is approximately symmetric")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Original scale
# Histogram
axes[0, 0].hist(df['AveragePrice_GBP'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].axvline(df['AveragePrice_GBP'].mean(), color='red', linestyle='--', linewidth=2, 
                   label=f"Mean: £{df['AveragePrice_GBP'].mean():,.0f}")
axes[0, 0].axvline(df['AveragePrice_GBP'].median(), color='green', linestyle='--', linewidth=2,
                   label=f"Median: £{df['AveragePrice_GBP'].median():,.0f}")
axes[0, 0].set_xlabel('Average Price (£)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Distribution (Original Scale)', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Box plot
axes[0, 1].boxplot(df['AveragePrice_GBP'], vert=True)
axes[0, 1].set_ylabel('Average Price (£)', fontsize=12)
axes[0, 1].set_title('Box Plot (Outlier Detection)', fontsize=14, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Q-Q plot
stats.probplot(df['AveragePrice_GBP'], dist="norm", plot=axes[0, 2])
axes[0, 2].set_title('Q-Q Plot vs Normal Distribution', fontsize=14, fontweight='bold')
axes[0, 2].grid(alpha=0.3)

# Row 2: Log-transformed scale
log_prices = np.log1p(df['AveragePrice_GBP'])

# Log histogram
axes[1, 0].hist(log_prices, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1, 0].axvline(log_prices.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {log_prices.mean():.2f}")
axes[1, 0].axvline(log_prices.median(), color='green', linestyle='--', linewidth=2, label=f"Median: {log_prices.median():.2f}")
axes[1, 0].set_xlabel('Log(Average Price)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Distribution (Log Scale)', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Log box plot
axes[1, 1].boxplot(log_prices, vert=True)
axes[1, 1].set_ylabel('Log(Average Price)', fontsize=12)
axes[1, 1].set_title('Box Plot (Log Scale)', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

# Log Q-Q plot
stats.probplot(log_prices, dist="norm", plot=axes[1, 2])
axes[1, 2].set_title('Q-Q Plot (Log Scale)', fontsize=14, fontweight='bold')
axes[1, 2].grid(alpha=0.3)

plt.suptitle('Target Variable Analysis: Original vs Log-Transformed', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n💡 Key Insight:")
print(f"   Original skewness: {df['AveragePrice_GBP'].skew():.4f}")
print(f"   Log-transformed skewness: {log_prices.skew():.4f}")
print("   → Log transformation significantly improves normality!")

## 4. Temporal Analysis

Understand how house prices evolved over time (2015-2025).

In [ ]:
# Convert Date to datetime for time series analysis
df['Date_dt'] = pd.to_datetime(df['Date'])

# Overall price trend over time
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 1. Average price over time
monthly_avg = df.groupby('Date_dt')['AveragePrice_GBP'].mean()
axes[0, 0].plot(monthly_avg.index, monthly_avg.values, linewidth=2, color='steelblue')
axes[0, 0].set_xlabel('Date', fontsize=12)
axes[0, 0].set_ylabel('Average Price (£)', fontsize=12)
axes[0, 0].set_title('Overall Price Trend (2015-2025)', fontsize=14, fontweight='bold')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Yearly average
yearly_avg = df.groupby('Year')['AveragePrice_GBP'].mean()
axes[0, 1].bar(yearly_avg.index, yearly_avg.values, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Year', fontsize=12)
axes[0, 1].set_ylabel('Average Price (£)', fontsize=12)
axes[0, 1].set_title('Yearly Average Price', fontsize=14, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Monthly seasonality
monthly_seasonal = df.groupby('Month')['AveragePrice_GBP'].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1, 0].plot(monthly_seasonal.index, monthly_seasonal.values, marker='o', linewidth=2, markersize=8, color='green')
axes[1, 0].set_xlabel('Month', fontsize=12)
axes[1, 0].set_ylabel('Average Price (£)', fontsize=12)
axes[1, 0].set_title('Seasonal Pattern (Average by Month)', fontsize=14, fontweight='bold')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(month_names)
axes[1, 0].grid(alpha=0.3)

# 4. Year-over-year growth rate
yoy_growth = yearly_avg.pct_change() * 100
colors = ['green' if x > 0 else 'red' for x in yoy_growth]
axes[1, 1].bar(yoy_growth.index[1:], yoy_growth.values[1:], color=colors, edgecolor='black', alpha=0.7)
axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1, 1].set_xlabel('Year', fontsize=12)
axes[1, 1].set_ylabel('YoY Growth (%)', fontsize=12)
axes[1, 1].set_title('Year-over-Year Price Growth', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Temporal Insights:")
print(f"   Overall price change (2015-2025): {((yearly_avg.iloc[-1] - yearly_avg.iloc[0]) / yearly_avg.iloc[0] * 100):.1f}%")
print(f"   Strongest year: {yearly_avg.idxmax()} (£{yearly_avg.max():,.0f})")
print(f"   Weakest year: {yearly_avg.idxmin()} (£{yearly_avg.min():,.0f})")

## 5. Regional Analysis

Compare house prices across UK regions.

In [ ]:
# Regional price comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Average price by region
regional_avg = df.groupby('Region')['AveragePrice_GBP'].mean().sort_values(ascending=True)
colors_region = ['darkred' if r == 'London' else 'steelblue' for r in regional_avg.index]

axes[0].barh(range(len(regional_avg)), regional_avg.values, color=colors_region, edgecolor='black', alpha=0.7)
axes[0].set_yticks(range(len(regional_avg)))
axes[0].set_yticklabels(regional_avg.index)
axes[0].set_xlabel('Average Price (£)', fontsize=12)
axes[0].set_title('Average House Price by Region', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add value labels
for i, (region, price) in enumerate(regional_avg.items()):
    axes[0].text(price + 5000, i, f'£{price:,.0f}', va='center', fontsize=10)

# 2. Box plot by region
df_sorted = df.copy()
df_sorted['Region'] = pd.Categorical(df_sorted['Region'], categories=regional_avg.index, ordered=True)
df_sorted = df_sorted.sort_values('Region')

axes[1].boxplot([df_sorted[df_sorted['Region'] == r]['AveragePrice_GBP'].values for r in regional_avg.index],
                vert=False, labels=regional_avg.index)
axes[1].set_xlabel('Average Price (£)', fontsize=12)
axes[1].set_title('Price Distribution by Region', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Regional Insights:")
print(f"   Most expensive region: {regional_avg.idxmax()} (£{regional_avg.max():,.0f})")
print(f"   Least expensive region: {regional_avg.idxmin()} (£{regional_avg.min():,.0f})")
print(f"   Price gap: £{regional_avg.max() - regional_avg.min():,.0f} ({(regional_avg.max() / regional_avg.min() - 1) * 100:.1f}% difference)")

In [ ]:
# Regional trends over time
fig, ax = plt.subplots(figsize=(16, 8))

for region in df['Region'].unique():
    region_data = df[df['Region'] == region].groupby('Date_dt')['AveragePrice_GBP'].mean()
    linewidth = 3 if region == 'London' else 1.5
    alpha = 1.0 if region == 'London' else 0.7
    ax.plot(region_data.index, region_data.values, label=region, linewidth=linewidth, alpha=alpha)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Average Price (£)', fontsize=12)
ax.set_title('Regional Price Trends Over Time', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n💡 Trend Insight: London (bold line) consistently commands premium prices across all periods.")

## 6. Property Type Analysis

Compare prices across different property types.

In [ ]:
# Property type comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Average price by property type
property_avg = df.groupby('PropertyType')['AveragePrice_GBP'].mean().sort_values(ascending=False)

axes[0].bar(property_avg.index, property_avg.values, color='coral', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Property Type', fontsize=12)
axes[0].set_ylabel('Average Price (£)', fontsize=12)
axes[0].set_title('Average Price by Property Type', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for i, (ptype, price) in enumerate(property_avg.items()):
    axes[0].text(i, price + 5000, f'£{price:,.0f}', ha='center', fontsize=11)

# 2. Distribution by property type
property_types = df['PropertyType'].unique()
positions = range(1, len(property_types) + 1)
data_to_plot = [df[df['PropertyType'] == pt]['AveragePrice_GBP'].values for pt in property_avg.index]

bp = axes[1].boxplot(data_to_plot, positions=positions, labels=property_avg.index, widths=0.6)
axes[1].set_xlabel('Property Type', fontsize=12)
axes[1].set_ylabel('Average Price (£)', fontsize=12)
axes[1].set_title('Price Distribution by Property Type', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Property Type Insights:")
print(f"   Most expensive: {property_avg.idxmax()} (£{property_avg.max():,.0f})")
print(f"   Least expensive: {property_avg.idxmin()} (£{property_avg.min():,.0f})")

## 7. Feature Correlations

Understand relationships between numeric features.

In [ ]:
# Select numeric features
numeric_features = ['Year', 'Month', 'AveragePrice_GBP', 'SalesVolume', 'MedianPrice_GBP', 'PricePerSqM_GBP']

# Correlation matrix
corr_matrix = df[numeric_features].corr()

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax,
            vmin=-1, vmax=1)

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Correlations with target
target_corr = corr_matrix['AveragePrice_GBP'].sort_values(ascending=False)
print("\n=" * 80)
print("CORRELATIONS WITH TARGET (AveragePrice_GBP)")
print("=" * 80)
print(target_corr)

print("\n💡 Key Insights:")
print(f"   Strongest positive correlation: {target_corr.index[1]} ({target_corr.iloc[1]:.3f})")
print(f"   Strongest negative correlation: {target_corr.index[-1]} ({target_corr.iloc[-1]:.3f})")
print("\n   Note: MedianPrice and PricePerSqM are highly correlated with AveragePrice")
print("   → These might be useful features, but watch for multicollinearity")

In [ ]:
# Scatter plots: Key features vs Target
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. MedianPrice vs AveragePrice
axes[0].scatter(df['MedianPrice_GBP'], df['AveragePrice_GBP'], alpha=0.5, s=20)
axes[0].set_xlabel('Median Price (£)', fontsize=12)
axes[0].set_ylabel('Average Price (£)', fontsize=12)
axes[0].set_title(f'Median vs Average Price\n(r = {corr_matrix.loc["MedianPrice_GBP", "AveragePrice_GBP"]:.3f})', 
                  fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Add regression line
z = np.polyfit(df['MedianPrice_GBP'], df['AveragePrice_GBP'], 1)
p = np.poly1d(z)
axes[0].plot(df['MedianPrice_GBP'], p(df['MedianPrice_GBP']), "r--", alpha=0.8, linewidth=2)

# 2. PricePerSqM vs AveragePrice
axes[1].scatter(df['PricePerSqM_GBP'], df['AveragePrice_GBP'], alpha=0.5, s=20, color='coral')
axes[1].set_xlabel('Price per m² (£)', fontsize=12)
axes[1].set_ylabel('Average Price (£)', fontsize=12)
axes[1].set_title(f'Price per m² vs Average Price\n(r = {corr_matrix.loc["PricePerSqM_GBP", "AveragePrice_GBP"]:.3f})',
                  fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

z = np.polyfit(df['PricePerSqM_GBP'], df['AveragePrice_GBP'], 1)
p = np.poly1d(z)
axes[1].plot(df['PricePerSqM_GBP'], p(df['PricePerSqM_GBP']), "r--", alpha=0.8, linewidth=2)

# 3. SalesVolume vs AveragePrice
axes[2].scatter(df['SalesVolume'], df['AveragePrice_GBP'], alpha=0.5, s=20, color='green')
axes[2].set_xlabel('Sales Volume', fontsize=12)
axes[2].set_ylabel('Average Price (£)', fontsize=12)
axes[2].set_title(f'Sales Volume vs Average Price\n(r = {corr_matrix.loc["SalesVolume", "AveragePrice_GBP"]:.3f})',
                  fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Multivariate Analysis

Explore interactions between multiple features.

In [ ]:
# Region × Property Type interaction
fig, ax = plt.subplots(figsize=(14, 8))

# Create pivot table
pivot_data = df.pivot_table(values='AveragePrice_GBP', index='Region', columns='PropertyType', aggfunc='mean')

# Sort by average price across all property types
pivot_data = pivot_data.loc[pivot_data.mean(axis=1).sort_values(ascending=False).index]

# Plot grouped bar chart
pivot_data.plot(kind='barh', ax=ax, width=0.8, edgecolor='black', alpha=0.8)

ax.set_xlabel('Average Price (£)', fontsize=12)
ax.set_ylabel('Region', fontsize=12)
ax.set_title('Average Price: Region × Property Type', fontsize=14, fontweight='bold')
ax.legend(title='Property Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Interaction Insights:")
print("   London detached properties are the most expensive segment")
print("   Property type hierarchy (Detached > Semi > Terraced > Flat) holds across most regions")
print("   Regional differences are more pronounced than property type differences")

## 9. Outlier Detection

Identify unusual observations in the target variable.

In [ ]:
# IQR-based outlier detection
Q1 = df['AveragePrice_GBP'].quantile(0.25)
Q3 = df['AveragePrice_GBP'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['AveragePrice_GBP'] < lower_bound) | (df['AveragePrice_GBP'] > upper_bound)]

print("=" * 80)
print("OUTLIER ANALYSIS (IQR Method)")
print("=" * 80)
print(f"\nLower bound: £{lower_bound:,.0f}")
print(f"Upper bound: £{upper_bound:,.0f}")
print(f"\nOutliers detected: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

if len(outliers) > 0:
    print(f"\nOutlier characteristics:")
    print(outliers.groupby(['Region', 'PropertyType']).size().sort_values(ascending=False).head(10))
    
    print("\n💡 Outlier Strategy:")
    print("   - Most outliers are likely London detached properties (legitimate high prices)")
    print("   - DON'T remove these - they're real market data")
    print("   - Log transformation will reduce their influence on the model")

## 10. Data Preprocessing Roadmap

Summary of findings and next steps for Phase 2.

In [ ]:
print("=" * 80)
print("PHASE 1 SUMMARY & PREPROCESSING ROADMAP")
print("=" * 80)

print("\n1. TARGET VARIABLE (AveragePrice_GBP):")
print(f"   - Skewness: {df['AveragePrice_GBP'].skew():.4f} (HIGHLY right-skewed)")
print("   ✓ ACTION: Apply log transformation (mandatory for linear regression)")
print(f"   - Outliers: {len(outliers)} detected ({len(outliers)/len(df)*100:.1f}%)")
print("   ✓ ACTION: Keep outliers (legitimate market data), log transform will handle")

print("\n2. TEMPORAL FEATURES:")
print("   - Time range: 2015-2025 (132 months)")
print(f"   - Overall price growth: {((yearly_avg.iloc[-1] - yearly_avg.iloc[0]) / yearly_avg.iloc[0] * 100):.1f}%")
print("   ✓ ACTION: Create time-based features (months since start, year dummies)")
print("   ✓ ACTION: Consider polynomial terms for non-linear time trends")

print("\n3. CATEGORICAL FEATURES:")
print(f"   - Region: {df['Region'].nunique()} categories")
print(f"   - PropertyType: {df['PropertyType'].nunique()} categories")
print("   ✓ ACTION: One-hot encode both features")
print("   ✓ ACTION: Create Region × PropertyType interaction terms")

print("\n4. NUMERIC FEATURES:")
print("   - MedianPrice_GBP: r = 0.996 (very strong correlation)")
print("   - PricePerSqM_GBP: r = 0.776 (strong correlation)")
print("   ✓ ACTION: Check for multicollinearity (VIF analysis)")
print("   ✓ ACTION: Consider dropping MedianPrice (too correlated with target)")
print("   ✓ ACTION: Standardize numeric features for regularized regression")

print("\n5. FEATURE ENGINEERING IDEAS:")
print("   ✓ Price momentum: (current_price - price_12mo_ago) / price_12mo_ago")
print("   ✓ Regional price rank (ordinal encoding by average price)")
print("   ✓ Interaction: Region × Year (capture regional growth rates)")
print("   ✓ Cyclical encoding: sin/cos transforms for Month (capture seasonality)")

print("\n6. TRAIN-TEST SPLIT STRATEGY:")
print("   ⚠ TIME SERIES DATA: Cannot use random split!")
print("   ✓ ACTION: Use temporal split (e.g., train on 2015-2023, test on 2024-2025)")
print("   ✓ ACTION: Implement time-series cross-validation for hyperparameter tuning")

print("\n" + "=" * 80)
print("NEXT STEPS: Phase 2 - Feature Engineering & Preprocessing")
print("=" * 80)
print("\n1. Transform target: log(AveragePrice_GBP)")
print("2. Create temporal features")
print("3. Encode categorical variables (one-hot)")
print("4. Handle multicollinearity (drop MedianPrice or use regularization)")
print("5. Create interaction terms")
print("6. Feature scaling (StandardScaler)")
print("7. Temporal train-test split")
print("8. Feature selection (if needed)")
print("\n" + "=" * 80)

## 11. Save Phase 1 Findings

In [ ]:
import pickle

# Save key findings for Phase 2
phase1_findings = {
    'target_skewness': df['AveragePrice_GBP'].skew(),
    'outliers_count': len(outliers),
    'outlier_indices': outliers.index.tolist(),
    'correlation_matrix': corr_matrix,
    'regional_avg_prices': regional_avg.to_dict(),
    'property_type_avg_prices': property_avg.to_dict(),
    'time_range': (df['Year'].min(), df['Year'].max()),
    'numeric_features': numeric_features,
    'categorical_features': ['Region', 'PropertyType']
}

with open('phase1_findings.pkl', 'wb') as f:
    pickle.dump(phase1_findings, f)

print("✓ Phase 1 findings saved to 'phase1_findings.pkl'")
print("\nLoad in Phase 2 with:")
print(">>> with open('phase1_findings.pkl', 'rb') as f:")
print(">>>     findings = pickle.load(f)")

---

## Phase 1 Complete! 🎉

**Key Discoveries:**
- ✓ Clean dataset (no missing values)
- ✓ Target is highly right-skewed → log transformation essential
- ✓ Strong temporal trends (11-year period)
- ✓ Significant regional variation (London premium)
- ✓ Property type hierarchy consistent across regions
- ✓ Time series structure → requires temporal train-test split

**Critical Insights for Linear Regression:**
1. **Must log-transform target** - skewness of 1.89 violates normality assumption
2. **Watch for multicollinearity** - MedianPrice correlates 0.996 with target
3. **Temporal structure matters** - can't use random CV, need time-series split
4. **Interactions are important** - Region × PropertyType shows clear patterns

**Next: Phase 2 - Feature Engineering**

You now have a solid foundation. In Phase 2, you'll:
- Transform the target variable
- Engineer temporal features
- Create interaction terms
- Handle the time series nature properly
- Build a preprocessing pipeline

Good luck! 🚀